# F.E.A.T.H.E.R. — the notebook

This notebook is the **code-based alternative to the FEATHER app**. It runs the exact
same pipeline, using the same tested modules (`playback_tools`, `harp_control`,
`harp_upload`, `feather_analysis`):

1. **Design** the randomized play order
2. **Load** the sounds onto the Harp Sound Card  *(hardware)*
3. **Run** the experiment  *(hardware — plays sound!)*
4. **Analyze** — detect onsets, identify each sound + its time offset (ms), check sync
5. **Spike-sort** with Kilosort4
6. **PSTH** and hand off to Phy

**No rig yet?** Skip parts 2–3, run the *"Generate demo data"* cell, then do parts 4–6
on a synthetic recording. Run cells top to bottom.

## Setup — run this first

In [ ]:
import os, sys, time, glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Find the FEATHER package folder (this notebook lives in notebooks/), so the modules import.
_p = os.getcwd()
while _p and not os.path.exists(os.path.join(_p, "playback_gui.py")):
    _p = os.path.dirname(_p)
PKG = _p
sys.path.insert(0, PKG)

import playback_tools as pt          # build the play order
import feather_analysis as fa        # onset detection, identity, sorting, PSTH

STIMULI_DIR = os.path.join(PKG, "stimuli")
OUTPUT_DIR  = os.path.join(PKG, "output"); os.makedirs(OUTPUT_DIR, exist_ok=True)
PROBE_FILE  = os.path.join(PKG, "probe_H5D.json")   # or probe_H15.json
CARD_SR     = 96000       # Harp card sample rate
GATE_OUTPUT = 0           # digital output wired to ADC1 (rig wiring; DO0)
print("Package:", PKG)
print("Stimuli:", STIMULI_DIR)

## 0a. Record vocalizations  *(hardware — needs a microphone)*

The notebook equivalent of the app's **Record** tab. Listens on each channel and saves a file
whenever a bird calls, starting a couple of seconds *before* the sound so onsets are never
clipped. Every file is then measured and kept only if it really is a vocalization; rejects are
moved to a `rejected/` folder with the reason, never deleted.

`Recorder` runs in the background — run the next cell, wait, then run the stop cell.


In [ ]:
import song_recorder as srec

print("Input devices:")
for i, name, ch, sr in srec.list_devices():
    print(f"  {i}: {name}  ({ch} in, {sr:g} Hz)")

SONG_DATA = os.path.join(PKG, "song_data")
rec = srec.Recorder(
    outdir=SONG_DATA,
    device=None,            # None = system default; use an index from the list above
    channels=1,             # one bird per channel
    fs=44100,
    bird_names=["bird1"],
    trigger=dict(threshold_db=-40.0, hold_s=1.0, prebuffer_s=2.0, min_events=2),
    accept=dict(enabled=True, min_syllables=3, min_bout_ms=150.0),
    on_event=lambda r: print(('kept    ' if r['accepted'] else 'rejected'),
                             r['file'], f"{r['duration_s']:.2f}s", r['reason']),
)
rec.start()
print("Recording. Run the next cell to stop.")


In [ ]:
rec.stop()
print(f"Kept {rec.n_kept}, rejected {rec.n_rejected}. Files under {SONG_DATA}")


## 0b. Measure calls and cut stimuli

The notebook equivalent of the app's **Sound Library** tab: measure every recording, look at
one, and cut a stretch of it into your stimulus folder — where step 1 below will find it.

Defaults are tuned for **chick begging calls** (analysis band up to 15 kHz), not adult song.


In [ ]:
import song_features as sfx, song_library as slib

SONG_DATA = os.path.join(PKG, "song_data")     # or any folder of WAVs
found = slib.scan_folder(SONG_DATA if os.path.isdir(SONG_DATA) else STIMULI_DIR)
print(found[["file", "bird", "day", "duration_s"]].to_string(index=False))

lib = slib.Library(os.path.join(PKG, slib.CACHE_NAME))   # cached, so re-runs are instant
table = lib.analyze_many(list(found["path"]))
print("\n", lib.summary(table).to_string(index=False))


In [ ]:
# Look at one recording: sonogram, a feature, and where the calls were found.
path = found["path"].iloc[0]
x, fs = slib.read_audio(path)
feat, syl, sp = sfx.analyze(x, fs)
print(f"{len(syl)} calls, median {syl['duration_ms'].median():.0f} ms")

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                       gridspec_kw=dict(height_ratios=[2, 1]))
img = np.abs(sp["deriv"]).T
ax[0].imshow(img, aspect="auto", origin="lower", cmap="Greys",
             vmin=0, vmax=np.percentile(img, 99.5),
             extent=[0, sp["t"][-1], sp["freqs"][0] / 1000, sp["freqs"][-1] / 1000])
ax[0].set(ylabel="kHz", title=os.path.basename(path))
ax[1].plot(feat["t"], feat["amplitude"], lw=0.8)
ax[1].set(xlabel="seconds", ylabel="amplitude (dB)")
for _, r in syl.iterrows():
    for a in ax:
        a.axvspan(r["start_s"], r["end_s"], color="#f59e0b", alpha=0.18)
plt.tight_layout(); plt.show()


In [ ]:
# Cut a call into the stimulus folder. It is faded, normalised, and resampled to match
# whatever rate that folder already uses, and the source and offsets are logged in
# stimulus_provenance.csv next to it.
first = syl.iloc[0]
out = slib.export_stimulus(path, STIMULI_DIR,
                           start_s=float(first["start_s"]) - 0.02,
                           end_s=float(first["end_s"]) + 0.02,
                           note="cut in the notebook")
print("wrote", out)
print(slib.provenance(STIMULI_DIR).tail(3).to_string(index=False))


## 0c. Generate begging calls  *(the VAE)*

The notebook equivalent of the app's **Generate Calls** tab. A conditional VAE trained on
32,647 real calls from chicks aged 4–15 days. Settings outside the training range are passed
through unclipped on purpose — that is how supernormal stimuli are made — and are flagged and
recorded when you save.

Runs on CPU in well under a second per call; no GPU needed.

In [ ]:
import song_generate as sgen

ok, why = sgen.available()
print("generator ready" if ok else why)

params = dict(sgen.defaults())
params.update(age_days=10, centroid_khz=3.9, duration_ms=105,
              calls_per_bout=6, n_bouts=1, ici_ms=150, seed=1)

y, specs = sgen.render(params)
print(f"{len(y)/sgen.SR:.2f} s, {len(specs)} calls")

# Measure it with FEATHER's own analysis — the same code tab 2 uses on real
# recordings, so these numbers are directly comparable to your own birds.
print(sgen.describe(y))
print("supernormal settings:", sgen.supernormal_settings(params) or "none")

In [ ]:
from IPython.display import Audio, display
import song_features as sfx

feat, syl, sp = sfx.analyze(y.astype(float), sgen.SR)
fig, ax = plt.subplots(figsize=(11, 3))
img = np.abs(sp["deriv"]).T
ax.imshow(img, aspect="auto", origin="lower", cmap="Greys",
          vmin=0, vmax=np.percentile(img, 99.5),
          extent=[0, sp["t"][-1], sp["freqs"][0]/1000, sp["freqs"][-1]/1000])
ax.set(ylabel="kHz", xlabel="seconds", title="generated call")
plt.tight_layout(); plt.show()
display(Audio(y, rate=sgen.SR))

In [ ]:
# Save it as a stimulus — from here it is an ordinary WAV, and step 1 will find it.
out = sgen.save_stimulus(y, STIMULI_DIR, params, note="generated in the notebook")
print("wrote", out)

# A graded set for a supernormal series: one stimulus per step, everything else
# (including the seed) held fixed. Brightness grades evenly; the F0 slider does not.
# outs = sgen.series(params, "centroid_khz", [2.5, 3.5, 4.5, 5.5, 6.5, 7.5],
#                    STIMULI_DIR, name_prefix="ladder")
# print(*outs, sep="\n")

## 1. Design the play order

`SETTINGS` is the only thing you normally edit. For **reproducibility**: a fixed
`seed` gives the *same* order+gaps every run; `seed=None` gives a fresh random order
each time. See `playback_tools.build_schedule` for every option.

In [ ]:
stim_table = pt.scan_stimuli(STIMULI_DIR)
print(stim_table.to_string(index=False), "\n")

SETTINGS = dict(
    repeats=5,                       # each sound plays this many times
    randomize_order=True,
    avoid_immediate_repeat=False,    # allow back-to-back repeats
    gap_mode="set",                  # gaps drawn from a discrete set...
    gap_set=[5, 10, 15, 20, 25],     # ...of these many seconds
    lead_in=5.0, tail=5.0,
    seed=42,                         # fixed = identical every run; None = different
)
schedule = pt.build_schedule(stim_table, **SETTINGS)
schedule.to_csv(os.path.join(OUTPUT_DIR, "schedule.csv"), index=False)
print(schedule.to_string(index=False))
print(f"\n{len(schedule)} presentations, total {schedule['offset_s'].iloc[-1]+schedule['gap_after_s'].iloc[-1]:.0f} s")

## 2. Load the sounds onto the card  *(hardware)*

Uploads each stimulus into a card memory slot (2, 3, 4, ...). Needs the manufacturer
TCP server (`server_v1.0.exe`) — the uploader auto-starts it. Skip this whole section
if you're just testing the analysis on demo data.

In [ ]:
import soundfile as sf
import harp_upload as hu

file_index = {f: 2 + i for i, f in enumerate(stim_table["file"])}   # sound -> card slot
for f in stim_table["file"]:
    x, sr = sf.read(os.path.join(STIMULI_DIR, f), dtype="float32", always_2d=True)
    wave  = hu.wav_to_card_wave_int(x[:, 0], sr, CARD_SR)           # -> 96 kHz stereo int32
    ok    = hu.upload_sound(wave, file_index[f], sample_rate=CARD_SR)
    print(("loaded " if ok else "FAILED ") + f"slot {file_index[f]}: {f}")
print("\nSounds are on the card." )

## 3. Run the experiment  *(hardware — THIS PLAYS SOUND)*

**Start the Open Ephys recording FIRST**, then run this cell. Make sure it is safe for
the animal to hear the stimuli. The card's gate (DO0) is set to go HIGH for each sound,
which the recording captures on ADC1 as the sample-accurate onset.

In [ ]:
import harp_control as hc

log = []
with hc.SoundCard() as sc:                      # opens COM4 (only ONE program at a time!)
    if sc.whoami()[0] != 1280:
        raise RuntimeError("Sound card not responding on COM4.")
    sc.configure_gate(GATE_OUTPUT, hc.DO_HIGH_WHEN_SOUND)
    t0 = time.perf_counter()
    for _, r in schedule.iterrows():
        while time.perf_counter() - t0 < r["onset_s"]:
            time.sleep(0.001)                   # wait until this trial's scheduled onset
        sc.play(file_index[r["file"]])          # AUDIBLE
        log.append(dict(trial=int(r["trial"]), file=r["file"],
                        sound_index=file_index[r["file"]],
                        onset_s=round(time.perf_counter() - t0, 4)))
        print(f"trial {int(r['trial']):>2}: {r['file']}")
    time.sleep(float(schedule['duration_s'].iloc[-1]) + 0.2)   # let the last sound finish

pd.DataFrame(log).to_csv(os.path.join(OUTPUT_DIR, "trigger_log.csv"), index=False)
print("\nDone. Stop the Open Ephys recording now.")

## No rig yet? — Generate demo data

Writes a **fake** 128-channel Open Ephys recording (with simulated neurons, a few of
which respond to the sounds) so you can try parts 4–6 with no hardware. It's a few
hundred MB. After it runs, `RECORDING` points at the synthetic session.

In [ ]:
import subprocess
subprocess.run([sys.executable, os.path.join(PKG, "make_test_data.py")], check=True)
RECORDING = fa.find_recording(os.path.join(PKG, "test_data"))
print("Demo recording:", RECORDING)

## 4. Analyze — onsets, identity, time offset, sync

Set `RECORDING` to your Open Ephys session folder (the one containing `continuous`),
or use the demo path from the cell above. This detects each sound (ADC1 gate), reads
its identity + **time offset in ms** from the ADC2 audio copy, and checks the sync.

In [ ]:
# RECORDING = r"C:\\Users\\you\\Documents\\Open Ephys\\2026-08-05_10-18-52\\Record Node 122\\experiment1\\recording1"
# (or keep the demo RECORDING from the cell above)
try:
    RECORDING
except NameError:
    RECORDING = fa.find_recording(os.path.expanduser(r"~\Documents\Open Ephys"))
print("Recording:", RECORDING)

rec = fa.load_adc_stream(RECORDING)
onsets, offsets = fa.detect_onsets(rec["data"], rec["names"], rec["fs"])
print(f"Length {rec['n']/rec['fs']:.1f}s -> {len(onsets)} sound starts, {len(offsets)} ends\n")

ident = fa.identify_by_audio(rec["data"], rec["names"], rec["fs"], onsets, STIMULI_DIR)
rows = []
for k, (o, (f, off, c)) in enumerate(zip(onsets, ident)):
    rows.append(dict(sound_number=k+1, onset_sample=int(o),
                     onset_seconds=round(o/rec["fs"], 6),
                     sound_file=f, audio_offset_ms=round(off, 3), audio_corr=round(c, 3)))
onset_df = pd.DataFrame(rows)
onset_df.to_csv(os.path.join(RECORDING, "sound_onsets.csv"), index=False)
onset_df.to_csv(os.path.join(OUTPUT_DIR, "sound_onsets_latest.csv"), index=False)
for _, r in onset_df.iterrows():
    print(f"  sound {int(r['sound_number']):2d}: {r['onset_seconds']:8.3f}s  "
          f"{r['sound_file'].split('_')[0]:8s}  {r['audio_offset_ms']:+.2f} ms vs marker")
o = onset_df["audio_offset_ms"]
print(f"\nAudio-vs-marker offset: mean {o.mean():+.2f} ms, SD {o.std():.2f} ms, "
      f"range [{o.min():+.2f}, {o.max():+.2f}] ms")

resid = fa.timing_check(onsets, offsets, rec["fs"], ident, STIMULI_DIR)
if len(resid):
    print(f"Gate width vs stimulus length: mean |{np.mean(np.abs(resid)):.3f}| ms, "
          f"jitter (SD) {np.std(resid):.3f} ms  "
          f"{'(sample-accurate)' if np.std(resid) < 1 else '(CHECK WIRING)'}")

## 5. Spike-sort with Kilosort4  *(GPU)*

Runs Kilosort4 on the recording's neural channels using `PROBE_FILE`. Output goes to
`<RECORDING>/kilosort4` (Phy-compatible). Uses the same fixed defaults as the app; pass
`params={...}` to override for tuning.

In [ ]:
ks_dir = fa.run_sorting(RECORDING, PROBE_FILE)      # device="cpu" if you have no GPU (slow)
print("Sorted ->", ks_dir)

## 6. Units and PSTH

In [ ]:
ks = fa.load_kilosort(ks_dir)
onsets = onset_df["onset_sample"].to_numpy()
print(f"{len(ks['units'])} units:")
for u in ks["units"]:
    n = int(np.sum(ks["spike_clusters"] == u))
    d = fa.responsiveness(ks, onsets, int(u))
    print(f"  unit {int(u):3d}: {n:5d} spikes   response {d:+.1f} Hz" + ("   <- responds" if d > 3 else ""))

fig = fa.psth(ks, int(ks["units"][0]), onsets)     # change the unit number as you like
plt.show()

## 7. Hand off to Phy (manual curation)

Kilosort4's output is already Phy-compatible. In a terminal:

```bat
conda activate phy2
cd <RECORDING>\kilosort4
phy template-gui params.py
```

That opens your sorted data in Phy for manual curation. When done, re-run parts 6+ —
`feather_analysis.load_kilosort` reads Phy's updated `spike_clusters.npy` automatically.

`sound_onsets.csv` (saved in the recording folder) has each sound's onset sample, its
identity, and its time offset — everything you need to align spikes to stimuli in your
own analysis.